<a href="https://colab.research.google.com/github/iria-param/visitor_analytics/blob/codex%2Fapproach-2-id-stability/approach-2-id-stability/notebooks/colab_tracker_comparison.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Museum Gallery AI - Colab Tracker Comparison

Use this notebook to compare baseline ByteTrack, museum ByteTrack, and museum BoT-SORT on real experience-center CCTV clips.

Privacy rules:
- Do not commit videos, overlays, or events logs to Git.
- Keep clips in a private Drive folder.
- Run with `--no-overlay` for the batch comparison.
- Delete `/content/videos` and `/content/runs` after downloading comparison CSV/JSON.
- No ReID, face recognition, demographic inference, emotion inference, or person crops.

## 1. Enable GPU

In Colab: `Runtime -> Change runtime type -> T4 GPU` if available. Then run the next cell.

In [ ]:
!nvidia-smi
!python - <<'PY'
import torch
print('torch cuda available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise SystemExit('GPU is not available. Change Colab runtime to GPU before continuing.')
PY

## 2. Mount Drive And Define Inputs

Upload the three CCTV clips to a private Google Drive folder first. Use stable file names with no spaces if possible.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Change this folder to your private Drive folder containing the CCTV clips.
DRIVE_VIDEO_DIR = '/content/drive/MyDrive/museum_gallery_ai_cctv'

# Edit these three clip entries after uploading the actual files.
CLIPS = [
    {'clip_id': 'gallery_day1', 'path': f'{DRIVE_VIDEO_DIR}/20260507140240.mp4'},
    {'clip_id': 'gallery_day2', 'path': f'{DRIVE_VIDEO_DIR}/20260507140651.mp4'},
    {'clip_id': 'gallery_day3', 'path': f'{DRIVE_VIDEO_DIR}/20260507140752.mp4'},
]

for clip in CLIPS:
    print(clip['clip_id'], clip['path'])

## 3. Clone And Install Project

Set `REPO_URL` to your GitHub repository URL if this project is pushed. If not, upload a zip of the project and adjust this cell.

In [ ]:
import os
from pathlib import Path

REPO_URL = 'https://github.com/iria-param/visitor_analytics.git'
BRANCH = 'codex/approach-2-id-stability'
PROJECT_DIR = Path('/content/museum-gallery-ai')

if REPO_URL == 'PASTE_YOUR_REPO_URL_HERE':
    raise ValueError('Set REPO_URL to the GitHub repository URL before continuing.')

if not PROJECT_DIR.exists():
    !git clone --branch {BRANCH} {REPO_URL} {PROJECT_DIR}
else:
    print('Project already exists:', PROJECT_DIR)

os.chdir(PROJECT_DIR)

!python -m pip install --upgrade pip
!python -m pip install -r requirements-dev.txt
!python -m pip install -e .
!python -m pytest -v

## 4. Generate Runtime Tracker Configs

These generated configs are runtime-only and should not be committed.

In [ ]:
import copy
import json
from pathlib import Path

base_config = json.loads(Path('configs/eval/tracker_only.json').read_text())
runtime_config_dir = Path('/content/runtime_configs')
runtime_config_dir.mkdir(parents=True, exist_ok=True)

TRACKERS = {
    'baseline': 'bytetrack.yaml',
    'bytetrack_museum': 'configs/trackers/bytetrack_museum.yaml',
    'botsort_museum': 'configs/trackers/botsort_museum.yaml',
}

for tracker_name, tracker_path in TRACKERS.items():
    config = copy.deepcopy(base_config)
    config['detector']['tracker'] = tracker_path
    config['detector']['device'] = 'cuda'
    config['detector']['image_size'] = 1280
    config['processing']['frame_stride'] = 1
    config['processing']['write_overlay'] = False
    out_path = runtime_config_dir / f'tracker_only_{tracker_name}.json'
    out_path.write_text(json.dumps(config, indent=2))
    print(tracker_name, out_path)

## 5. Run Batch Comparison

This runs 3 clips x 3 trackers. It writes `_done.txt` after each successful run so you can rerun the cell after a reconnect.

In [ ]:
import subprocess
import time
from pathlib import Path

runs_dir = Path('/content/runs')
runs_dir.mkdir(parents=True, exist_ok=True)

for clip in CLIPS:
    source = Path(clip['path'])
    if not source.exists():
        raise FileNotFoundError(source)
    for tracker_name in TRACKERS:
        run_name = f"{clip['clip_id']}_{tracker_name}"
        output_dir = runs_dir / run_name
        done_file = output_dir / '_done.txt'
        if done_file.exists():
            print('Skipping completed run:', run_name)
            continue
        output_dir.mkdir(parents=True, exist_ok=True)
        config_path = runtime_config_dir / f'tracker_only_{tracker_name}.json'
        command = [
            'python', '-m', 'museum_gallery_ai', 'process',
            '--config', str(config_path),
            '--source', str(source),
            '--output', str(output_dir),
            '--no-overlay',
        ]
        print('Running:', ' '.join(command))
        started = time.perf_counter()
        subprocess.run(command, check=True)
        done_file.write_text(f"completed in {time.perf_counter() - started:.2f}s\n")
        print('Completed:', run_name)

print('Batch complete.')

## 6. Build Comparison CSV And JSON

In [ ]:
comparison_csv = runs_dir / 'comparison_real_cctv.csv'
!python scripts/compare_runs.py --runs {runs_dir} --out {comparison_csv} --expected-runs 9

import pandas as pd
df = pd.read_csv(comparison_csv)
display(df)
display(df.groupby('tracker')[['unique_track_count', 'duration_median', 'short_lived_count', 'likely_switch_count', 'max_gap_processed_frames']].median())

## 7. Download Only Comparison Outputs

In [ ]:
from google.colab import files

files.download(str(runs_dir / 'comparison_real_cctv.csv'))
files.download(str(runs_dir / 'comparison_real_cctv.json'))

## 8. Cleanup Runtime Files

Run this after downloading the comparison outputs.

In [ ]:
!rm -rf /content/runs /content/runtime_configs
print('Deleted runtime runs/configs from Colab VM. Disconnect and delete runtime next.')